### Esercizio 1
Hai a disposizione un file `data.csv` contenente dati mensili di traffico aereo con due colonne:

- `date`: data in formato `YYYY-MM` (mese/anno)
- `passengers`: numero di passeggeri per quel mese


Costruisci un modello di **regressione polinomiale** che approssima l’andamento del numero di passeggeri nel tempo.

1. Carica il dataset.
2. Convertilo in un formato numerico utilizzando una colonna `mese_numerico` che conti i mesi a partire da gennaio 1949.
3. Applica una regressione polinomiale (grado a tua scelta).
4. Calcola l’RMSE tra i valori reali e quelli predetti.
5. Visualizza i dati reali e la curva stimata con Plotly.

## 1. CELLA DEI CALCOLI

#### LEGGENDA

- **pd.read_csv()** –> carica il dataset

- **pd.to_datetime()** –> converte la colonna ***date*** in datetime

- **df['mese_numerico'] = (df['date'].dt.year - 1949) * 12 + df['date'].dt.month - 1** – crea una variabile numerica progressiva a partire da gennaio 1949 (mese 0)

- **PolynomialFeatures(degree=3)** – Genera le feature polinomiali (x, x², x³, …)

- **LinearRegression()** –> Modello di regressione lineare (sulle feature polinomiali)

- **root_mean_squared_error(y_true, y_pred)** –> Calcola l’RMSE (radice dell’errore quadratico medio)

- **plotly.graph_objects.Scatter()** –> Grafico con dati reali (punti) e curva predetta (linea)

- **fig.update_layout()** –> Titoli, Etichette

- **go.Figure()** –> Figura plotly

In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import root_mean_squared_error
import plotly.graph_objects as go

# 1. Caricamento dataset
df = pd.read_csv("data.csv")


# 2. Conversione date e mese numerico
df['date'] = pd.to_datetime(df['date'])
df['mese_numerico'] = (df['date'].dt.year - 1949) * 12 + (df['date'].dt.month - 1)


# 3. Pulizia (rimozione NaN)
df['passengers'] = pd.to_numeric(df['passengers'], errors='coerce')
df_clean = df.dropna().copy()


# 4. Variabili
X = df_clean[['mese_numerico']]
y = df_clean['passengers']


# 5. Regressione polinomiale grado 3
grado = 3
poly = PolynomialFeatures(degree=grado)
X_poly = poly.fit_transform(X)
model = LinearRegression()
model.fit(X_poly, y)
y_pred = model.predict(X_poly)


# 6. RMSE
rmse = root_mean_squared_error(y, y_pred)


# 7. Preparazione dati per il grafico (ordinati per data)
df_ordered = df_clean.sort_values('date')
y_pred_ordered = model.predict(poly.transform(df_ordered[['mese_numerico']]))



# Print finale
print(f"RMSE tra valori reali e predetti: {rmse:.2f} passeggeri")



RMSE tra valori reali e predetti: 44.46 passeggeri


## 2. CELLA DEI GRAFICI

In [6]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_clean['date'],
    y=y,
    mode='markers',
    name='Dati reali',
    marker=dict(color='blue', size=4)
))
fig.add_trace(go.Scatter(
    x=df_ordered['date'],
    y=y_pred_ordered,
    mode='lines',
    name=f'Polinomio grado {grado}',
    line=dict(color='red', width=2)
))
fig.update_layout(
    title="Andamento passeggeri – regressione polinomiale",
    xaxis_title="Data",
    yaxis_title="Numero di passeggeri",
    template="plotly_white"
)
fig.show()